# Paso 1: Cargar el dataset, verificar esquema y primeras filas (3.3 - 3.4)

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("sesion2-fundamentos-spark")
    .master("local[*]")
    .config("spark.ui.port", "4040")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/28 13:09:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
ORIGEN_DATOS = "/opt/data"
ARTIFACTS = "/opt/artifacts"

In [3]:
from pyspark.sql.types import StructType, StructField, LongType, DoubleType, BooleanType
schema_trades = StructType([
    StructField("trade_id", LongType(), True),
    StructField("price", DoubleType(), True),
    StructField("qty", DoubleType(), True),
    StructField("quote_qty", DoubleType(), True),
    StructField("time", LongType(), True),
    StructField("is_buyer_maker", BooleanType(), True),
    StructField("is_best_match", BooleanType(), True)
])

In [4]:
df_trades = (
    spark.read
    .option("header", "false")
    .schema(schema_trades)
    .csv(f"{ORIGEN_DATOS}/BTCUSDT-trades-2026-01-05.csv")
)

In [5]:
df_trades.printSchema()

root
 |-- trade_id: long (nullable = true)
 |-- price: double (nullable = true)
 |-- qty: double (nullable = true)
 |-- quote_qty: double (nullable = true)
 |-- time: long (nullable = true)
 |-- is_buyer_maker: boolean (nullable = true)
 |-- is_best_match: boolean (nullable = true)



In [7]:
df_trades.show(5, truncate=False)

+----------+--------+-------+-----------+----------------+--------------+-------------+
|trade_id  |price   |qty    |quote_qty  |time            |is_buyer_maker|is_best_match|
+----------+--------+-------+-----------+----------------+--------------+-------------+
|5734054604|91529.74|2.2E-4 |20.1365428 |1767571200308618|false         |true         |
|5734054605|91529.74|0.01   |915.2974   |1767571200375801|false         |true         |
|5734054606|91529.74|0.00437|399.9849638|1767571200477157|false         |true         |
|5734054607|91529.74|0.00764|699.2872136|1767571200480543|false         |true         |
|5734054608|91529.74|0.00136|124.4804464|1767571200545508|false         |true         |
+----------+--------+-------+-----------+----------------+--------------+-------------+
only showing top 5 rows


# Paso 2: Transformaciones, acción y evaluación perezosa (3.5)

In [6]:
# Transformación 1: Filtrar trades significativos (monto > 1000 USDT)
from pyspark.sql.functions import col
df_filtrado = df_trades.filter(col("quote_qty") > 1000)

In [7]:
df_filtrado.show(5, truncate=False)

+----------+--------+-------+-------------+----------------+--------------+-------------+
|trade_id  |price   |qty    |quote_qty    |time            |is_buyer_maker|is_best_match|
+----------+--------+-------+-------------+----------------+--------------+-------------+
|5734054612|91529.74|0.11112|10170.7847088|1767571201085463|false         |true         |
|5734054613|91529.74|0.08092|7406.5865608 |1767571201085463|false         |true         |
|5734054614|91529.74|0.02555|2338.584857  |1767571201085463|false         |true         |
|5734054620|91529.74|0.58846|53861.5908004|1767571202033686|false         |true         |
|5734054624|91529.74|0.20378|18651.9304172|1767571202033686|false         |true         |
+----------+--------+-------+-------------+----------------+--------------+-------------+
only showing top 5 rows


In [8]:
# Transformación 2: Seleccionar columnas clave para el análisis
df_seleccionado = df_filtrado.select("trade_id", "price", "qty", "quote_qty", "is_buyer_maker")

In [9]:
df_seleccionado.show(5, truncate=False)

+----------+--------+-------+-------------+--------------+
|trade_id  |price   |qty    |quote_qty    |is_buyer_maker|
+----------+--------+-------+-------------+--------------+
|5734054612|91529.74|0.11112|10170.7847088|false         |
|5734054613|91529.74|0.08092|7406.5865608 |false         |
|5734054614|91529.74|0.02555|2338.584857  |false         |
|5734054620|91529.74|0.58846|53861.5908004|false         |
|5734054624|91529.74|0.20378|18651.9304172|false         |
+----------+--------+-------+-------------+--------------+
only showing top 5 rows


In [10]:
# Transformación 3: Crear columna indicadora de orden grande
df_transformado = df_seleccionado.withColumn("es_ballena", col("quote_qty") > 50000)

In [11]:
df_transformado.show(5, truncate=False)

+----------+--------+-------+-------------+--------------+----------+
|trade_id  |price   |qty    |quote_qty    |is_buyer_maker|es_ballena|
+----------+--------+-------+-------------+--------------+----------+
|5734054612|91529.74|0.11112|10170.7847088|false         |false     |
|5734054613|91529.74|0.08092|7406.5865608 |false         |false     |
|5734054614|91529.74|0.02555|2338.584857  |false         |false     |
|5734054620|91529.74|0.58846|53861.5908004|false         |true      |
|5734054624|91529.74|0.20378|18651.9304172|false         |false     |
+----------+--------+-------+-------------+--------------+----------+
only showing top 5 rows


In [12]:
print("Objeto DataFrame creado (sin ejecutar aún):", df_transformado)

Objeto DataFrame creado (sin ejecutar aún): DataFrame[trade_id: bigint, price: double, qty: double, quote_qty: double, is_buyer_maker: boolean, es_ballena: boolean]


In [13]:
total_registros_grandes = df_transformado.count()
print(f"Total de operaciones > 1000 USDT: {total_registros_grandes}")

[Stage 3:=====>                                                    (1 + 9) / 10]

Total de operaciones > 1000 USDT: 242078


# Paso 3: Analizar el plan de ejecución con explain() (3.6)

In [14]:
df_transformado.explain(True)

== Parsed Logical Plan ==
'Project [unresolvedstarwithcolumns(es_ballena, '`>`('quote_qty, 50000), None)]
+- Project [trade_id#0L, price#1, qty#2, quote_qty#3, is_buyer_maker#5]
   +- Filter (quote_qty#3 > cast(1000 as double))
      +- Relation [trade_id#0L,price#1,qty#2,quote_qty#3,time#4L,is_buyer_maker#5,is_best_match#6] csv

== Analyzed Logical Plan ==
trade_id: bigint, price: double, qty: double, quote_qty: double, is_buyer_maker: boolean, es_ballena: boolean
Project [trade_id#0L, price#1, qty#2, quote_qty#3, is_buyer_maker#5, (quote_qty#3 > cast(50000 as double)) AS es_ballena#58]
+- Project [trade_id#0L, price#1, qty#2, quote_qty#3, is_buyer_maker#5]
   +- Filter (quote_qty#3 > cast(1000 as double))
      +- Relation [trade_id#0L,price#1,qty#2,quote_qty#3,time#4L,is_buyer_maker#5,is_best_match#6] csv

== Optimized Logical Plan ==
Project [trade_id#0L, price#1, qty#2, quote_qty#3, is_buyer_maker#5, (quote_qty#3 > 50000.0) AS es_ballena#58]
+- Filter (isnotnull(quote_qty#3) AND (

# Paso 4: Aplicar funciones sobre columnas (withColumn, when, lit, .cast()) (3.7)

In [15]:
from pyspark.sql.functions import when, lit, from_unixtime

df_trades_enriquecido = (
    df_trades
    # Convierte timestamp de microsegundos a segundos y lo castea a timestamp
    .withColumn("fecha_hora", (col("time") / 1000000).cast("timestamp"))
    # Categorización según si el comprador fue creador de mercado (Maker vs Taker)
    .withColumn("tipo_orden", when(col("is_buyer_maker") == True, "Venta Taker").otherwise("Compra Taker"))
    # Columna constante de identificación del par de criptomonedas
    .withColumn("par", lit("BTCUSDT"))
)

In [16]:
df_trades_enriquecido.select("trade_id", "price", "qty", "fecha_hora", "tipo_orden", "par").show(5)

+----------+--------+-------+--------------------+------------+-------+
|  trade_id|   price|    qty|          fecha_hora|  tipo_orden|    par|
+----------+--------+-------+--------------------+------------+-------+
|5734054604|91529.74| 2.2E-4|2026-01-05 00:00:...|Compra Taker|BTCUSDT|
|5734054605|91529.74|   0.01|2026-01-05 00:00:...|Compra Taker|BTCUSDT|
|5734054606|91529.74|0.00437|2026-01-05 00:00:...|Compra Taker|BTCUSDT|
|5734054607|91529.74|0.00764|2026-01-05 00:00:...|Compra Taker|BTCUSDT|
|5734054608|91529.74|0.00136|2026-01-05 00:00:...|Compra Taker|BTCUSDT|
+----------+--------+-------+--------------------+------------+-------+
only showing top 5 rows


# Paso 5: Agrupación y agregación relevante (3.9)

In [17]:
from pyspark.sql.functions import sum, avg, count, max, min

df_resumen_mercado = (
    df_trades_enriquecido
    .groupBy("tipo_orden")
    .agg(
        count("trade_id").alias("total_operaciones"),
        sum("qty").alias("volumen_total_btc"),
        sum("quote_qty").alias("volumen_total_usdt"),
        avg("price").alias("precio_promedio"),
        min("price").alias("precio_minimo"),
        max("price").alias("precio_maximo")
    )
)

In [18]:
df_resumen_mercado.show(truncate=False)

[Stage 7:>                                                        (0 + 10) / 10]

+------------+-----------------+------------------+--------------------+-----------------+-------------+-------------+
|tipo_orden  |total_operaciones|volumen_total_btc |volumen_total_usdt  |precio_promedio  |precio_minimo|precio_maximo|
+------------+-----------------+------------------+--------------------+-----------------+-------------+-------------+
|Compra Taker|2978348          |10892.969499996278|1.0158587519317032E9|93398.43814111676|91514.82     |94789.08     |
|Venta Taker |2507141          |9780.626339999657 |9.11596491616359E8  |93329.75343975221|91514.81     |94789.07     |
+------------+-----------------+------------------+--------------------+-----------------+-------------+-------------+



# Paso 6: Convertir a RDD y aplicar operaciones bajo el modelo MapReduce (3.10)

In [19]:
rdd_trades = df_trades.rdd

In [20]:
rdd_altos = rdd_trades.filter(lambda row: row.price > 91529.74)

In [21]:
rdd_kv = rdd_altos.map(lambda row: (row.is_buyer_maker, row.qty))

In [22]:
rdd_volumen_por_rol = rdd_kv.reduceByKey(lambda a, b: a + b)

In [23]:
resultados_rdd = rdd_volumen_por_rol.collect()

In [24]:
for es_maker, vol_btc in resultados_rdd:
    rol = "Maker es Comprador" if es_maker else "Maker es Vendedor"
    print(f"Rol: {rol} | Volumen acumulado BTC: {vol_btc:.4f}")

Rol: Maker es Vendedor | Volumen acumulado BTC: 10887.9828
Rol: Maker es Comprador | Volumen acumulado BTC: 9779.6150
